In [1]:
from cwtraces import sca101_lab_data
import chipwhisperer as cw
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import trange


In [2]:
data = sca101_lab_data["lab3_3"]()
trace_array =  data["trace_array"]
textin_array = data["textin_array"]

assert(len(trace_array) == 2500)

In [3]:
#known key (this is just to check if the attack works properly as this data is already known from lesson)
#we would not do this for an actual attack as we dont know real data

In [4]:
known_key = [0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6, 0xab, 0xf7, 0x15, 0x88, 0x09, 0xcf, 0x4f, 0x3c]

In [5]:
#S-box

In [6]:
sbox = [
    # 0    1    2    3    4    5    6    7    8    9    a    b    c    d    e    f 
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76, # 0
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0, # 1
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15, # 2
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75, # 3
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84, # 4
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf, # 5
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8, # 6
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2, # 7
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73, # 8
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb, # 9
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79, # a
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08, # b
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a, # c
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e, # d
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf, # e
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16  # f
]

def aes_internal(inputdata, key):
    return sbox[inputdata ^ key]

In [7]:
#gives back full difference curve across all sample points from bit=1 and bit=0 the trace groups for 1 key guess and byte posistion

In [8]:
numtraces = trace_array.shape[0]
samples_per_trace = trace_array.shape[1]

In [9]:
def calculate_diffs(guess, byte_idx=0):
    #does a DPA on two traces, useing`textin_array` and `trace_array` 
    
    list_one = []
    list_zero = []

    for trace_idx in range(numtraces):
        output_hyp = aes_internal(guess, textin_array[trace_idx][byte_idx])

      
        if output_hyp & 0x01:
            list_one.append(trace_array[trace_idx])
        else:
            list_zero.append(trace_array[trace_idx])

    avg_one = np.asarray(list_one).mean(axis=0)
    avg_zero = np.asarray(list_zero).mean(axis=0)
    return np.abs(avg_one - avg_zero)

In [10]:
#plot correct guess vs wrong guesses for comparing

In [11]:
cw.plot(calculate_diffs(0x2B)) * cw.plot(calculate_diffs(0x2C)) * cw.plot(calculate_diffs(0x2D))

:Overlay
   .Curve.I   :Curve   [x]   (y)
   .Curve.II  :Curve   [x]   (y)
   .Curve.III :Curve   [x]   (y)

In [12]:
#attacks each of 16 key bytes using DPA(trying all 256 possible byte vals), builds recovered key, compares it to real key (this is the attack)

In [14]:
key_recovered = []
winning_scores = []
for byte_idx in trange(16, desc="attacking byte"):
    max_diffs = np.zeros(256)

    for guess in range(256):
        diff_curve = calculate_diffs(guess, byte_idx)
        max_diffs[guess] = np.max(diff_curve)
    best_guess = np.argmax(max_diffs)
    key_recovered.append(best_guess)
    winning_scores.append(max_diffs[best_guess]) 

    compare = "True" if best_guess == known_key[byte_idx] else "false"
    print (f"byte {byte_idx}: guess={best_guess:02x} actual={known_key[byte_idx]:02x} diff={max_diffs[best_guess]:.6f} {compare}")

    key_recovered = bytearray(key_recovered)
    print("Best Key Guess: ", end="")
    for b in key_recovered:
        print("%02x " % b, end="")
    print()
    print("Actual key: ", end="")
    for b in known_key:
        print("%02x " % b, end="")
    print()
    print("Match:", key_recovered == bytearray(known_key))
    

attacking byte:   0%|          | 0/16 [00:00<?, ?it/s]

byte 0: guess=2b actual=2b diff=0.002540 True
Best Key Guess: 2b 
Actual key: 2b 7e 15 16 28 ae d2 a6 ab f7 15 88 09 cf 4f 3c 
Match: False
byte 1: guess=7e actual=7e diff=0.002435 True
Best Key Guess: 2b 7e 
Actual key: 2b 7e 15 16 28 ae d2 a6 ab f7 15 88 09 cf 4f 3c 
Match: False
byte 2: guess=15 actual=15 diff=0.002381 True
Best Key Guess: 2b 7e 15 
Actual key: 2b 7e 15 16 28 ae d2 a6 ab f7 15 88 09 cf 4f 3c 
Match: False
byte 3: guess=16 actual=16 diff=0.002679 True
Best Key Guess: 2b 7e 15 16 
Actual key: 2b 7e 15 16 28 ae d2 a6 ab f7 15 88 09 cf 4f 3c 
Match: False
byte 4: guess=28 actual=28 diff=0.002188 True
Best Key Guess: 2b 7e 15 16 28 
Actual key: 2b 7e 15 16 28 ae d2 a6 ab f7 15 88 09 cf 4f 3c 
Match: False
byte 5: guess=ae actual=ae diff=0.002379 True
Best Key Guess: 2b 7e 15 16 28 ae 
Actual key: 2b 7e 15 16 28 ae d2 a6 ab f7 15 88 09 cf 4f 3c 
Match: False
byte 6: guess=d2 actual=d2 diff=0.002674 True
Best Key Guess: 2b 7e 15 16 28 ae d2 
Actual key: 2b 7e 15 16 28 ae d

In [15]:
cw.plot(winning_scores)

:Curve   [x]   (y)